# Self-Reflection & Critique Practical

This notebook builds a **writing critic agent** that improves a draft through a controlled loop:

```text
generate or load draft → critique with rubric → store reflection → refine → stop or retry
```

Earlier prompting work focused on getting a model to produce useful outputs. This practical adds a second skill: designing a system that can **judge and improve** an output before it is accepted.

By the end, you will have a reusable evaluator-generator loop that can later critique agent outputs such as travel itineraries, summaries, RAG answers, or code reviews.

## Practical architecture

The system has three roles:

1. **Generator** — creates a first draft or accepts a weak draft.
2. **Critic** — scores the draft using a rubric and principles.
3. **Refiner** — rewrites the draft using structured critique.

The code owns the stopping rule. The model can recommend whether something passes, but the application enforces the threshold.

In [1]:
from pathlib import Path

from critic_loop import (
    critique_draft,
    generate_first_draft,
    print_iteration_trace,
    read_text,
    refine_draft,
    run_refinement_loop,
    save_demo_outputs,
)
from llm_client import check_api_ready, load_settings

DATA_DIR = Path("data")
settings = load_settings()

print(f"Model: {settings.model}")
print(f"Offline demo mode: {settings.offline_demo}")

# TRAINER NOTE: If the classroom has API/network issues, set BIA_OFFLINE_DEMO=true in .env.
check_api_ready(settings)

Model: gpt-4o-mini
Offline demo mode: False


## Load the task brief, weak draft, rubric, and constitution

The critic needs more than “is this good?” It needs explicit judgment criteria.

- The **brief** defines the target output.
- The **weak draft** gives us a realistic imperfect starting point.
- The **rubric** turns quality into observable criteria.
- The **constitution** adds principles that prevent unsafe or misleading revisions.

In [2]:
brief = read_text(DATA_DIR / "writing_brief.txt")
weak_draft = read_text(DATA_DIR / "weak_draft.txt")
rubric = read_text(DATA_DIR / "rubric.md")
constitution = read_text(DATA_DIR / "constitution.md")

print("BRIEF\n", brief)
print("\nWEAK DRAFT\n", weak_draft)

BRIEF
 Write a short launch email for working professionals who are considering a live weekend program in Generative AI and Agentic AI Development.

Audience:
- Software engineers, data analysts, DevOps professionals, and tech managers
- They are busy, skeptical of hype, and want proof that the program is practical

Constraints:
- 140–180 words
- Warm but professional tone
- Mention live expert-led classes, hands-on projects, and career relevance
- Do not make unrealistic salary promises
- End with one clear call to action

WEAK DRAFT
 Hi everyone,

AI is changing the world and you should learn it as soon as possible. Our course will make you skilled in all AI tools and help you get better opportunities. You will study Generative AI and Agentic AI with experienced trainers and projects. This is a great chance to upgrade yourself.

Join now to become future ready.


## Step 1 — Critique the weak draft

This first call demonstrates the evaluator role.

Notice that the output is structured. That structure matters because it allows code to:

- display issues cleanly,
- retry only when needed,
- route severe failures differently,
- store a compact reflection memory.

In [3]:
# COST NOTE: One short evaluator call with gpt-4o-mini is typically a few cents or less.
first_critique = critique_draft(
    brief=brief,
    draft=weak_draft,
    rubric=rubric,
    constitution=constitution,
    pass_score=8,
)

print(first_critique.model_dump_json(indent=2))

{
  "score": 3,
  "passed": false,
  "summary": "The draft fails to meet several key criteria, particularly in specificity and audience fit.",
  "strengths": [],
  "issues": [
    {
      "criterion": "Audience fit",
      "severity": "major",
      "explanation": "The tone is overly enthusiastic and lacks the professionalism expected by busy working professionals.",
      "revision_instruction": "Adopt a more professional tone that acknowledges the audience's skepticism and busy schedules."
    },
    {
      "criterion": "Specificity",
      "severity": "major",
      "explanation": "The draft makes vague claims about becoming skilled in AI tools and better opportunities without detailing specific program benefits.",
      "revision_instruction": "Include concrete details about the live expert-led classes and hands-on projects that will be part of the program."
    },
    {
      "criterion": "Constraint following",
      "severity": "minor",
      "explanation": "The draft is too sh

### Read the critique like a developer

A good evaluator response has four layers:

- **Score** — useful for automation.
- **Issues** — useful for targeted repair.
- **Reflection memory** — useful for the next attempt.
- **Revised strategy** — useful for guiding the rewrite.

The important design choice: do not ask the model for a hidden chain of thought. Ask for **actionable evaluation fields**.

In [4]:
print(f"Score: {first_critique.score}/10")
print(f"Passed threshold? {first_critique.passed}")
print("\nTop revision instructions:")
for issue in first_critique.issues:
    print(f"- {issue.criterion} [{issue.severity}]: {issue.revision_instruction}")

print("\nReflection memory:")
print(first_critique.reflection_memory)

Score: 3/10
Passed threshold? False

Top revision instructions:
- Audience fit [major]: Adopt a more professional tone that acknowledges the audience's skepticism and busy schedules.
- Specificity [major]: Include concrete details about the live expert-led classes and hands-on projects that will be part of the program.
- Constraint following [minor]: Expand the content to meet the 140-180 word requirement while maintaining clarity and focus.
- Evidence quality [major]: Avoid making unsupported claims and focus on the practical skills and knowledge participants will gain.
- Call to action [minor]: Provide a specific action, such as 'Register now for our upcoming session to secure your spot.'

Reflection memory:
Ensure that the tone and content are tailored to the audience's professional context and skepticism.


## Step 2 — Refine using the critique

The refiner does not receive a vague instruction like “make it better.” It receives:

- the original brief,
- the current draft,
- the structured critique,
- the principles that must remain true,
- prior reflection memory.

In [5]:
# COST NOTE: One refiner call. Cost grows with longer drafts and critique history.
revised_draft = refine_draft(
    brief=brief,
    draft=weak_draft,
    critique=first_critique,
    constitution=constitution,
    reflections=[first_critique.reflection_memory],
)

print(revised_draft)

Subject: Elevate Your Skills in Generative and Agentic AI

Dear Professionals,

As the landscape of technology evolves, staying ahead is crucial. We invite you to join our live weekend program focused on Generative AI and Agentic AI Development. This course is designed specifically for busy professionals like you, offering expert-led classes and hands-on projects that will enhance your practical skills.

Throughout the program, you will engage with industry experts and work on real-world applications, ensuring that you gain relevant knowledge that can be applied directly to your career. Our curriculum is tailored to address the challenges you face in your roles, providing you with tools and insights that are immediately useful.

Don’t miss this opportunity to invest in your future. Register now for our upcoming session to secure your spot and take the next step in your professional development.

Best regards,  
[Your Name]  
[Your Position]  
[Your Contact Information]


## Step 3 — Critique the revised draft

Now evaluate the revised version using the same rubric and principles. This is where the loop becomes measurable.

In [6]:
second_critique = critique_draft(
    brief=brief,
    draft=revised_draft,
    rubric=rubric,
    constitution=constitution,
    reflections=[first_critique.reflection_memory],
    pass_score=8,
)

print(second_critique.model_dump_json(indent=2))

{
  "score": 7,
  "passed": false,
  "summary": "The draft is well-structured but lacks specificity and a clear call to action.",
  "strengths": [
    "The tone is warm and professional, suitable for the audience.",
    "The draft mentions expert-led classes and hands-on projects."
  ],
  "issues": [
    {
      "criterion": "Specificity",
      "severity": "major",
      "explanation": "The draft includes vague claims about gaining 'relevant knowledge' and 'tools and insights' without specifying what those are.",
      "revision_instruction": "Include specific examples of skills or tools participants will learn and how they apply to their roles."
    },
    {
      "criterion": "Call to action",
      "severity": "major",
      "explanation": "The call to action is not clear and lacks urgency; it simply states to 'register now' without a specific link or deadline.",
      "revision_instruction": "Provide a direct link to the registration page and mention a deadline for registration to

## Step 4 — Automate the loop

The full loop repeats until:

- the score reaches the passing threshold,
- a blocking issue disappears,
- or the maximum number of iterations is reached.

This is the core reusable pattern for self-reflection and critique.

In [7]:
# COST NOTE: This loop performs 1 critique call per iteration and 1 refine call after failed critiques.
# Keep max_iterations low in classroom demos.
final_draft, trace = run_refinement_loop(
    brief=brief,
    initial_draft=weak_draft,
    rubric=rubric,
    constitution=constitution,
    pass_score=8,
    max_iterations=3,
)

print_iteration_trace(trace)
print("\nFINAL DRAFT\n")
print(final_draft)

Iteration 1: score=3/10 | passed=False
Summary: The draft fails to meet several key criteria, particularly in specificity and audience fit.
Reflection memory: Focus on providing specific, concrete benefits and maintaining a professional tone for working professionals.
Top issues:
  - [major] Audience fit: Adopt a more professional tone that acknowledges the audience's skepticism and busy schedules.
  - [major] Specificity: Include specific benefits of the program, such as the types of projects and skills that will be developed.
  - [minor] Constraint following: Ensure the email is between 140-180 words and follows a clear structure with an introduction, body, and conclusion.
  - [major] Evidence quality: Avoid making broad claims and instead focus on the practical aspects of the program.
  - [minor] Call to action: Provide a clear and actionable next step, such as signing up for a webinar or visiting a website for more information.
------------------------------------------------------

## Step 5 — Save the trace

Saving the trace makes the loop inspectable. In production, this same trace can feed observability, evaluation dashboards, or human review.

In [8]:
save_demo_outputs(final_draft, trace)
print("Saved:")
print("- demo_outputs/final_draft.txt")
print("- demo_outputs/iteration_trace.json")

Saved:
- demo_outputs/final_draft.txt
- demo_outputs/iteration_trace.json


## Step 6 — Compare before and after

Use this comparison to discuss whether the loop genuinely improved the output or merely made it sound more polished.

In [9]:
print("BEFORE\n")
print(weak_draft)
print("\n" + "=" * 100 + "\n")
print("AFTER\n")
print(final_draft)

BEFORE

Hi everyone,

AI is changing the world and you should learn it as soon as possible. Our course will make you skilled in all AI tools and help you get better opportunities. You will study Generative AI and Agentic AI with experienced trainers and projects. This is a great chance to upgrade yourself.

Join now to become future ready.


AFTER

Subject: Enhance Your Skills in Generative AI and Agentic AI Development

Dear Professionals,

As AI technology evolves, it's essential to stay informed and skilled. We invite you to our live weekend program on Generative AI and Agentic AI Development, designed specifically for busy professionals like you.

Led by industry experts, our program focuses on practical applications through hands-on projects. You'll work with tools such as TensorFlow and PyTorch to build and deploy AI models, analyze data, and seamlessly integrate AI solutions into your workflows. By the end of the program, you will have developed a portfolio of projects that demo

## Challenge 1 — Make the critic stricter

Try changing the passing score from `8` to `9`.

Observe:

- Does the loop take more iterations?
- Does the final draft become better or just longer?
- At what point does extra refinement stop helping?

In [ ]:
# Try this after the main demo:
# stricter_final, stricter_trace = run_refinement_loop(
#     brief=brief,
#     initial_draft=weak_draft,
#     rubric=rubric,
#     constitution=constitution,
#     pass_score=9,
#     max_iterations=3,
# )
# print_iteration_trace(stricter_trace)
# print(stricter_final)

## Challenge 2 — Bridge to the travel planner project

The same pattern can critique itinerary quality.

Replace the writing brief with the bridge brief and design a travel-planner rubric:

- budget realism,
- feasible timing,
- personalization,
- logical ordering,
- clear trade-offs.

This prepares the evaluator component for the upcoming multi-agent travel planner build.

In [ ]:
# travel_bridge = read_text(DATA_DIR / "travel_planner_bridge_brief.txt")
# print(travel_bridge)

# # Extension idea:
# # 1. Write an intentionally weak itinerary.
# # 2. Create a travel-specific rubric.
# # 3. Reuse critique_draft() and refine_draft().

Upcoming project bridge:

The same evaluator-generator loop can critique a travel itinerary.

Example itinerary criteria:
- Budget realism
- Logical ordering of activities
- Time feasibility
- Personalization to user preferences
- Clear explanation of trade-offs


## Optional AutoGen evaluator-agent preview

AutoGen appears here only as a preview. The deep framework coverage comes later. The point is to see that the evaluator role can be wrapped as an agent while the underlying pattern remains the same.

Run the next cell only after installing:

```bash
pip install -r requirements_autogen.txt
```

In [11]:
# Optional cell. Skip if AutoGen is not installed.
# In a regular .py script, wrap this in asyncio.run(...).

import asyncio
from autogen_evaluator_preview import run_autogen_evaluator_preview

autogen_result = await run_autogen_evaluator_preview(
    brief=brief,
    draft=weak_draft,
    rubric=rubric,
    constitution=constitution,
)
print(autogen_result)

{
  "score": 4,
  "summary": "The draft lacks specificity and fails to address the audience's skepticism. It does not provide concrete benefits or a clear call to action, and the tone is overly enthusiastic without sufficient professionalism.",
  "issues": [
    "Vague claims about becoming 'skilled in all AI tools' without specifics.",
    "Lacks mention of live expert-led classes and hands-on projects.",
    "Does not address career relevance or practical applications.",
    "Tone is overly enthusiastic and lacks professionalism.",
    "No clear, actionable next step provided."
  ],
  "revision_instructions": [
    "Specify the benefits of live expert-led classes and hands-on projects.",
    "Include concrete examples of how the program is relevant to their careers.",
    "Adopt a more professional tone while maintaining warmth.",
    "Provide a clear call to action, such as signing up for a webinar or visiting a website for more information.",
    "Ensure the word count is within th

## Wrap-up

You built a practical reflection loop with:

- rubric-based critique,
- structured evaluator output,
- principles-based correction,
- reflection memory,
- threshold-based stopping,
- optional evaluator-agent wrapping.

The key lesson: self-reflection is useful only when it is designed as a controlled system, not as a loose prompt asking the model to “think again.”